<a href="https://colab.research.google.com/github/HORIZONrar/SPK-DATAMINING_KELOMPOK10/blob/main/DATA_MINING_%26_SPK_KELOMPOK_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fase 1 — Business Understanding**

Latar Belakang

Harga sepeda motor bekas dapat berbeda-beda berdasarkan merek, model, tahun produksi, jarak tempuh, dan riwayat kepemilikan. Perbedaan tersebut membuat estimasi harga jual menjadi tidak mudah.

Permasalahan

Bagaimana memprediksi harga jual sepeda motor bekas berdasarkan karakteristik kendaraan menggunakan metode Data Mining?

Tujuan

Membangun model machine learning yang mampu memprediksi harga jual sepeda motor bekas berdasarkan data historis kendaraan.

**Fase 2 — Data Understanding**

Pada fase ini, kita memahami isi dan karakteristik dataset BIKE DETAILS.csv sebelum melakukan preprocessing.

1. Membaca Dataset


In [ ]:
import pandas as pd

df = pd.read_csv("BIKE DETAILS.csv")

df.head()

,name,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
0,Royal Enfield Classic 350,175000,2019,Individual,1st owner,350,NaN
1,Honda Dio,45000,2017,Individual,1st owner,5650,NaN
2,Royal Enfield Classic Gunmetal Grey,150000,2018,Individual,1st owner,12000,148114.0
3,Yamaha Fazer FI V 2.0 [2016-2018],65000,2015,Individual,1st owner,23000,89643.0
4,Yamaha SZ [2013-2014],20000,2011,Individual,2nd owner,21000,NaN


Kode ini digunakan untuk membaca dataset dan menampilkan 5 baris pertama.

2. Memeriksa Struktur Dataset

In [ ]:
print("Jumlah baris dan kolom:", df.shape)
df.info()

Jumlah baris dan kolom: (1061, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1061 entries, 0 to 1060
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               1061 non-null   object 
 1   selling_price      1061 non-null   int64  
 2   year               1061 non-null   int64  
 3   seller_type        1061 non-null   object 
 4   owner              1061 non-null   object 
 5   km_driven          1061 non-null   int64  
 6   ex_showroom_price  626 non-null    float64
dtypes: float64(1), int64(3), object(3)
memory usage: 58.2+ KB


Dataset memiliki 1.061 baris dan 7 kolom.
| Kolom | Keterangan |
|---|---|
| `name` | Nama atau model motor |
| `selling_price` | Harga jual motor |
| `year` | Tahun produksi |
| `seller_type` | Jenis penjual |
| `owner` | Riwayat kepemilikan |
| `km_driven` | Jarak tempuh |
| `ex_showroom_price` | Harga motor saat baru |

Catatan: Harga pada dataset menggunakan mata uang India (INR).

3. Memeriksa Statistik Deskriptif

In [ ]:
df.describe(include="all")

,name,selling_price,year,seller_type,owner,km_driven,ex_showroom_price
count,1061,1061.000000,1061.000000,1061,1061,1061.000000,6.260000e+02
unique,279,NaN,NaN,2,4,NaN,NaN
top,Bajaj Pulsar 150,NaN,NaN,Individual,1st owner,NaN,NaN
freq,41,NaN,NaN,1055,924,NaN,NaN
mean,NaN,59638.151744,2013.867107,NaN,NaN,34359.833176,8.795871e+04
std,NaN,56304.291973,4.301191,NaN,NaN,51623.152702,7.749659e+04
min,NaN,5000.000000,1988.000000,NaN,NaN,350.000000,3.049000e+04
25%,NaN,28000.000000,2011.000000,NaN,NaN,13500.000000,5.485200e+04
50%,NaN,45000.000000,2015.000000,NaN,NaN,25000.000000,7.275250e+04
75%,NaN,70000.000000,2017.000000,NaN,NaN,43000.000000,8.703150e+04


Kode ini menampilkan statistik data, seperti rata-rata, nilai minimum, maksimum, dan distribusi kategori.

4. Memeriksa Missing Value dan Duplikasi

In [ ]:
print("Missing Value:")
print(df.isnull().sum())

print("\nJumlah Duplikasi:", df.duplicated().sum())

Missing Value:
name                   0
selling_price          0
year                   0
seller_type            0
owner                  0
km_driven              0
ex_showroom_price    435
dtype: int64

Jumlah Duplikasi: 6


Pemeriksaan ini bertujuan mengidentifikasi masalah kualitas data sebelum memasuki fase Data Preparation.

**Fase 3 — Data Preparation**

Pada fase ini, kita membersihkan dan menyiapkan dataset BIKE DETAILS.csv agar dapat digunakan untuk pelatihan model.

1. Data cleaning

In [ ]:
df = pd.read_csv("BIKE DETAILS.csv")
df = df.drop_duplicates().copy()

print("Jumlah data:", df.shape)
print("\nMissing Value:")
print(df.isnull().sum())

Jumlah data: (1055, 7)

Missing Value:
name                   0
selling_price          0
year                   0
seller_type            0
owner                  0
km_driven              0
ex_showroom_price    433
dtype: int64


ditemukan 6 duplikat dan dihapus, sehingga tersisa 1.055 data

2. Feature Engineering

Kita menambahkan fitur brand berdasarkan nama motor.

In [ ]:
def extract_brand(name):
    name = name.lower().strip()

    if name.startswith("royal enfield"):
        return "Royal Enfield"
    if name.startswith("hero honda"):
        return "Hero Honda"
    if name.startswith("harley davidson"):
        return "Harley-Davidson"
    if name.startswith("activa"):
        return "Honda"

    return name.split()[0].title()

df["brand"] = df["name"].apply(extract_brand)

df[["name", "brand"]].head()

,name,brand
0,Royal Enfield Classic 350,Royal Enfield
1,Honda Dio,Honda
2,Royal Enfield Classic Gunmetal Grey,Royal Enfield
3,Yamaha Fazer FI V 2.0 [2016-2018],Yamaha
4,Yamaha SZ [2013-2014],Yamaha


3. Menentukan Fitur dan Target

In [ ]:
X = df.drop(columns=["selling_price"])
y = df["selling_price"]

print("Fitur:", X.columns.tolist())
print("Target:", y.name)

Fitur: ['name', 'year', 'seller_type', 'owner', 'km_driven', 'ex_showroom_price', 'brand']
Target: selling_price


Fitur yang digunakan:

name, year, seller_type, owner, km_driven, ex_showroom_price, dan brand.

4. Membagi Data Training dan Testing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (844, 7)
Testing: (211, 7)


5. Menangani Missing Value dan Encoding

Kita menggunakan Pipeline agar imputasi dan encoding dipelajari dari data training saja.

Untuk ex_showroom_price, kita gunakan median berdasarkan name, kemudian brand, dan terakhir median keseluruhan.

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

class PriceImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()

        self.name_median_ = X.groupby("name")["ex_showroom_price"].median()
        self.brand_median_ = X.groupby("brand")["ex_showroom_price"].median()
        self.global_median_ = X["ex_showroom_price"].median()

        return self

    def transform(self, X):
        X = X.copy()

        missing = X["ex_showroom_price"].isna()

        X.loc[missing, "ex_showroom_price"] = (
            X.loc[missing, "name"].map(self.name_median_)
        )

        missing = X["ex_showroom_price"].isna()

        X.loc[missing, "ex_showroom_price"] = (
            X.loc[missing, "brand"].map(self.brand_median_)
        )

        X["ex_showroom_price"] = (
            X["ex_showroom_price"].fillna(self.global_median_)
        )

        return X

preprocessor = Pipeline([
    ("imputer", PriceImputer()),
    ("encoder", ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"),
         ["name", "seller_type", "brand"]),

        ("ordinal", OrdinalEncoder(
            categories=[[
                "1st owner",
                "2nd owner",
                "3rd owner",
                "4th owner"
            ]]
        ), ["owner"])
    ], remainder="passthrough"))
])

Penjelasan: Nilai kosong pada ex_showroom_price diisi berdasarkan karakteristik motor, sedangkan fitur kategorikal diubah menjadi numerik.

Imputasi dan encoding baru akan dijalankan ketika model dilatih melalui Pipeline, sehingga data testing tidak digunakan untuk menghitung nilai pengganti.

**Fase 4 — Modeling**

Pada fase ini, kita melatih dan membandingkan tiga algoritma regresi untuk memprediksi harga sepeda motor bekas:
1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Kita menggunakan hasil Data Preparation Fase 3, termasuk fitur brand dan ex_showroom_price yang ditangani menggunakan imputasi.

1. Training Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

trained_models = {}

for name, algorithm in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", algorithm)
    ])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    print(f"{name}: Training selesai")

Linear Regression: Training selesai
Decision Tree: Training selesai
Random Forest: Training selesai


Penjelasan:

Ketiga algoritma dilatih menggunakan data training yang sama agar hasil perbandingannya adil. Proses imputasi dan encoding dilakukan secara otomatis melalui Pipeline.

2. Evaluasi dan Perbandingan Model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2 Score": r2_score(y_test, y_pred)
    })

df_results = pd.DataFrame(results)

df_results.sort_values(by="MAE")

,Model,MAE,RMSE,R2 Score
2,Random Forest,12718.629510,33149.258351,0.748637
1,Decision Tree,14426.075829,30015.108202,0.793921
0,Linear Regression,15578.576010,29377.192289,0.802588


Penjelasan metrik:
- MAE: Rata-rata kesalahan prediksi harga. Semakin kecil semakin baik.
- RMSE: Mengukur kesalahan prediksi dengan penalti lebih besar untuk kesalahan ekstrem. Semakin kecil semakin baik.
- R² Score: Mengukur kemampuan model menjelaskan variasi harga. Semakin mendekati 1 semakin baik.

3. Pemeriksaan Overfitting

Selanjutnya, kita memeriksa apakah ketiga model mengalami overfitting dengan membandingkan performa pada data training dan testing.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

hasil_overfitting = []

for name, model in trained_models.items():

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    hasil_overfitting.append({
        "Model": name,
        "MAE Train": mean_absolute_error(y_train, pred_train),
        "MAE Test": mean_absolute_error(y_test, pred_test),
        "RMSE Train": np.sqrt(mean_squared_error(y_train, pred_train)),
        "RMSE Test": np.sqrt(mean_squared_error(y_test, pred_test)),
        "R2 Train": r2_score(y_train, pred_train),
        "R2 Test": r2_score(y_test, pred_test)
    })

df_overfitting = pd.DataFrame(hasil_overfitting)

display(df_overfitting.round(4))

,Model,MAE Train,MAE Test,RMSE Train,RMSE Test,R2 Train,R2 Test
0,Linear Regression,13059.5401,15578.5760,20087.1108,29377.1923,0.8592,0.8026
1,Decision Tree,33.1754,14426.0758,497.6247,30015.1082,0.9999,0.7939
2,Random Forest,4202.5274,12718.6295,9037.7434,33149.2584,0.9715,0.7486


Tujuan Pemeriksaan

Kita membandingkan hasil training dan testing untuk mengetahui apakah model mampu melakukan prediksi pada data yang belum pernah dilihat.
- MAE dan RMSE: Semakin kecil, semakin baik.
- R²: Semakin tinggi, semakin baik.
- Overfitting: Terindikasi ketika performa training jauh lebih baik daripada testing.

Analisis
1. Linear Regression

Memiliki perbedaan performa training dan testing paling kecil. Model ini menunjukkan kemampuan generalisasi yang relatif baik.

2. Decision Tree

Memiliki R² training hampir sempurna (0,9999), tetapi turun menjadi 0,7939 pada testing. Ini menunjukkan overfitting yang sangat kuat.

3. Random Forest

Memiliki MAE testing paling rendah (12.718,63), tetapi terdapat selisih cukup besar antara performa training dan testing.

Langkah Selanjutnya

Kita lakukan 5-Fold Cross-Validation untuk membandingkan kestabilan ketiga algoritma pada beberapa pembagian data.

In [ ]:
from sklearn.model_selection import cross_validate
import pandas as pd

hasil_cv = []

for name, model in trained_models.items():
    cv = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    hasil_cv.append({
        "Model": name,
        "MAE CV": -cv["test_MAE"].mean(),
        "RMSE CV": -cv["test_RMSE"].mean(),
        "R2 CV": cv["test_R2"].mean(),
        "Std MAE": cv["test_MAE"].std()
    })

df_cv = pd.DataFrame(hasil_cv)
display(df_cv.round(4))

,Model,MAE CV,RMSE CV,R2 CV,Std MAE
0,Linear Regression,13828.1581,21166.0283,0.8039,1760.8019
1,Decision Tree,15170.8484,31417.5271,0.5718,2778.6456
2,Random Forest,12041.0048,23618.8100,0.7924,3196.0902


Analisis
1. Linear Regression
- Memiliki RMSE terendah dan R² tertinggi.
- Standar deviasi MAE paling kecil, sehingga kesalahannya relatif lebih konsisten antar-fold.
- Menunjukkan kemampuan generalisasi yang baik.

2. Decision Tree
- Memiliki performa cross-validation paling rendah.
- Hasil ini memperkuat indikasi overfitting dari pengujian sebelumnya.

3. Random Forest
- Memiliki MAE terendah, yaitu 12.041,00.
- Namun, variasi MAE antar-fold lebih besar dibandingkan Linear Regression.

Kesimpulan
Berdasarkan hasil testing, pemeriksaan overfitting, dan cross-validation:
- Random Forest unggul dalam meminimalkan rata-rata kesalahan absolut harga (MAE).
- Linear Regression unggul dalam RMSE, R², dan konsistensi performa.
- Decision Tree tidak dipilih karena menunjukkan overfitting yang kuat.